# Example Notebook for the Pandas Units Extension

## Imports

In [1]:
import astropy.units as u
import numpy as np
import pandas as pd
import pandas_units_extension as pue

## Creating a unit-aware `pandas` `Series` or `DataFrame`

To create a unit-aware `pandas` `Series` or `DataFrame` from a `astropy` `Quantity` one just simply has to specify the `dtype` to `"unit"`.

### From `Quantity` objects

In [2]:
# Create a Quantity object of three values with the unit of meters
q = [1, 2, 3] * u.m
q

<Quantity [1., 2., 3.] m>

In [3]:
# Create a pandas Series with the UnitsDtype, the unit itself is derived from the Quantity object
length_sr: pd.Series = pd.Series(q, dtype="unit")
length_sr

0    1.0 m
1    2.0 m
2    3.0 m
dtype: unit[m]

In [4]:
# Get the dtype of the Series
length_sr.dtype

UnitsDtype("m")

In [5]:
# Create a DataFrame with a length and a time column, both with units
df: pd.DataFrame = pd.DataFrame(
    {
        "length": [1, 2, 3] * u.m,
        "time": [4, 5, 6] * u.s,
    }, dtype="unit"
)
df

,length,time
0,1.0 m,4.0 s
1,2.0 m,5.0 s
2,3.0 m,6.0 s


In [6]:
# Both of the Series of the DataFrame have the UnitsDtype, but with different units attached to them
df.dtypes

length    unit[m]
time      unit[s]
dtype: object

### From python native lists with setting the `unit` through the `dtype` argument

It is also possible to set the unit of a Series from the string representation of the unit, e.g., `m` for meters, by combining it with the base `unit` followed by the unit in square parentheses.

In [7]:
# Creating the same series as above, but from a list and string notaion of the dtype
pd.Series([1, 2, 3], dtype="unit[m]")

0    1.0 m
1    2.0 m
2    3.0 m
dtype: unit[m]

### From a file

The `dtype` can also be set when reading in a file, for example a `.csv` file:

In [8]:
# Read a CSV file containing the height of some mountains in feet.
# This uses the UnitsDtype by specifying the dtype for the height column as "unit[ft]".
peaks_df: pd.DataFrame = pd.read_csv("peaks.csv", dtype={"height": "unit[ft]",})
peaks_df

,name,height
0,Mount Elbert,14440.0 ft
1,Mount Massive,14428.0 ft
2,Mount Harvard,14421.0 ft
3,Blanca Peak,14351.0 ft
4,La Plata Peak,14343.0 ft
5,Uncompahgre Peak,14321.0 ft
6,Crestone Peak,14300.0 ft
7,Mount Lincoln,14293.0 ft
8,Castle Peak,14279.0 ft
9,Grays Peak,14278.0 ft


## Comparison and Arithmetic Operations

The unit-aware `pandas` objects can be used in many ways as native `Quantity` objects with the difference that the result will be packed back to `pandas` objects.

### Comparison Operations with zerodim `Quantity` object

In this example the height of the mountains in the `peaks_df` `DataFrame` will be compared to the height of the "Dent Blanche" and considered `tall` if taller. Notably the height in the `DataFrame` is in the unit of `feet` whereas the heigth of the comparison is measured in `meters`. This however is not an issue, as in the background the values are first converted to the same unit before conducting the comparison.

In [9]:
# Define height of the mountain used as comparison 
dent_blanche: u.Quantity = 4357 * u.m

# Create a boolean column that is True if the height of the mountain is greater than the height of the Dent Blanche
peaks_df["tall"] = peaks_df["height"] > dent_blanche
peaks_df

,name,height,tall
0,Mount Elbert,14440.0 ft,True
1,Mount Massive,14428.0 ft,True
2,Mount Harvard,14421.0 ft,True
3,Blanca Peak,14351.0 ft,True
4,La Plata Peak,14343.0 ft,True
5,Uncompahgre Peak,14321.0 ft,True
6,Crestone Peak,14300.0 ft,True
7,Mount Lincoln,14293.0 ft,False
8,Castle Peak,14279.0 ft,False
9,Grays Peak,14278.0 ft,False


### Arithmetic Operation with zerodim/scalar objects

Zerodim or scalar objects can be Python scalars (e.g. `0`, `1` or `10`) or a `numpy` `array` or `Quantity` object with `obj.ndim == 0` (e.g. `np.array(1)` or `1 * u.m`).

#### Arithmetic Operation with zerodim `Quantity` object

In this example a `DataFrame` of temperatures for different cities is defined and then a temeperature increase is calculated through a arithmetic operation with a zerodim `Quantity` object.

In [10]:
# Define a DataFrame with the temperatures of some cities in °C
temperature_df: pd.DataFrame = pd.DataFrame({
    "city": ["Prague", "Kathmandu", "Catania", "Boston"],
    "temperature": pd.Series([20, 22, 31, 16], dtype="unit[deg_C]"),
})
temperature_df.set_index("city", inplace=True)
temperature_df

,temperature
city,
Prague,20.0 deg_C
Kathmandu,22.0 deg_C
Catania,31.0 deg_C
Boston,16.0 deg_C


In [11]:
# Define a temperature increase of 10 degree Celsius
temperature_increase: u.Quantity = 10 * u.deg_C

# Create a new column in the DataFrame that is the result of an arithmethic operation between the temeperature column and the temperature increase
temperature_df["increased"] = temperature_df["temperature"] + temperature_increase
temperature_df

,temperature,increased
city,,
Prague,20.0 deg_C,30.0 deg_C
Kathmandu,22.0 deg_C,32.0 deg_C
Catania,31.0 deg_C,41.0 deg_C
Boston,16.0 deg_C,26.0 deg_C


#### Arithmetic operations with python scalars

Arithmetic operations with python scalars are possible as long as `Quantity` also supports them. As an example we will calculate the circumference and area of circles.

In [12]:
# Define a Series of radii of some circles in mm.
radii_sr: pd.Series = pd.Series([10, 20, 30], dtype="unit[mm]", index=["circle_A", "circle_B", "circle_C"])
radii_sr

circle_A    10.0 mm
circle_B    20.0 mm
circle_C    30.0 mm
dtype: unit[mm]

In [13]:
# Calculate the circumference of the circles using the formula C = 2 * pi * r. The resulting Series will have the unit of mm.
2 * np.pi * radii_sr

circle_A     62.83185307179586 mm
circle_B    125.66370614359172 mm
circle_C    188.49555921538757 mm
dtype: unit[mm]

In [14]:
# Calculate the area of the circles using the formula A = pi * r^2. The resulting Series will have the unit of mm^2.
np.pi * radii_sr**2

circle_A     314.1592653589793 mm2
circle_B    1256.6370614359173 mm2
circle_C    2827.4333882308138 mm2
dtype: unit[mm2]

### Arithmetic operations between `Series`

In this example a `DataFrame` containing sizes of different packages is defined and then the volume of each package is calculated from the x, y and z dimensions.

In [15]:
# Define a DataFrame with the length, width and height of some packages in cm
packages_df: pd.DataFrame = pd.DataFrame({
        "length": [50, 100, 150] * u.cm,
        "width": [40, 30, 50] * u.cm,
        "height": [3, 4, 5] * u.cm,
    },
    dtype="unit", 
    index=["package_1", "package_2", "package_3"]
)
packages_df

,length,width,height
package_1,50.0 cm,40.0 cm,3.0 cm
package_2,100.0 cm,30.0 cm,4.0 cm
package_3,150.0 cm,50.0 cm,5.0 cm


In [16]:
# Calculate the volume by multiplying the length, width and height columns together. The resulting column will have the unit of cm^3.
packages_df["volume"] = packages_df["length"] * packages_df["width"] * packages_df["height"]
packages_df

,length,width,height,volume
package_1,50.0 cm,40.0 cm,3.0 cm,6000.0 cm3
package_2,100.0 cm,30.0 cm,4.0 cm,12000.0 cm3
package_3,150.0 cm,50.0 cm,5.0 cm,37500.0 cm3


## The `units` `Series` and `DataFrame` Accessor and Conversions to another `Unit`

With the `units` `Series` and `DataFrame` accessor it is possible to access to the underlying `Quantity` objects and to convert them to a different `unit`.

### `UnitsSeriesAccessor`

Access the `UnitsSeriesAccessor` of a `Series` `sr` with `sr.units`. Then use one of its attributes or methods to access the underlying data or perform a conversion.

In [17]:
# Get the unit of the Series
length_sr.units.unit

Unit("m")

In [18]:
# Get the data of a Series as a Quantity object
length_sr.units.to_quantity()

<Quantity [1., 2., 3.] m>

In [19]:
# Convert the length Series to centimeters
length_sr.units.to("cm")

0    100.0 cm
1    200.0 cm
2    300.0 cm
dtype: unit[cm]

In [20]:
# The Pandas Units Extension has the `u.temperature()` equivalencies activated by default, so we can convert the temperature Series to Fahrenheit
temperature_df["temperature"].units.to("deg_F")

city
Prague                    68.0 deg_F
Kathmandu                 71.6 deg_F
Catania      87.80000000000001 deg_F
Boston                    60.8 deg_F
Name: temperature, dtype: unit[deg_F]

In [21]:
# It is also possible to give the `to` method your own equivalencies, like the spectral equivalency to convert between wavelength and frequency
length_sr.units.to(u.MHz, equivalencies=u.spectral())

0           299.792458 MHz
1           149.896229 MHz
2    99.93081933333332 MHz
dtype: unit[MHz]

In [22]:
# Convert a Series with the unit of km/h to its SI units of m/s
pd.Series([1, 2, 3], dtype="unit[km/h]").units.to_si()

0    0.2777777777777778 m / s
1    0.5555555555555556 m / s
2    0.8333333333333334 m / s
dtype: unit[m / s]

### `UnitsDataFrameAccessor`

Access the `UnitsDataFrameAccessor` of a `DataFrame` `df` with `df.units`. At the moment this has only the `to_si` method.

In [23]:
# `to_si` can also be used on a DataFrame, in which case it will convert all columns with UnitsDtype to their SI units and leave the other columns unchanged.
# As an example in the peaks DataFrame, only the height column will be converted (feet -> meters).
peaks_df.units.to_si()

,name,height,tall
0,Mount Elbert,4401.312 m,True
1,Mount Massive,4397.6544 m,True
2,Mount Harvard,4395.5208 m,True
3,Blanca Peak,4374.1848 m,True
4,La Plata Peak,4371.7464 m,True
5,Uncompahgre Peak,4365.040800000001 m,True
6,Crestone Peak,4358.64 m,True
7,Mount Lincoln,4356.5064 m,False
8,Castle Peak,4352.2392 m,False
9,Grays Peak,4351.9344 m,False


## Reductions

The `UnitsExtensionArray` supports the following numeric reductions: `min`, `max`, `sum`, `mean`, `std`, `var`, `median`, `sem`. They can be used on `Series`, `DataFrame` or during grouping (see ["Running Diary: Advanced Calculcations, Groupings and Reductions"](#running-diary-advanced-calculcations-groupings-and-reductions) for an example on that).

In [24]:
# Get min and max of the height column
print(f"Minimum height: {peaks_df['height'].min()}")
print(f"Mean height: {peaks_df['height'].mean()}")
print(f"Maximum height: {peaks_df['height'].max()}")

Minimum height: 14115.0 ft
Mean height: 14274.45 ft
Maximum height: 14440.0 ft


With `idxmin` and `idxmax` it is possible to access the respective index of the `Series`, as an example here of the `height` column. That allows to get the full row of the peak with the minimum height.

In [25]:
# Get index of the minimum height and use it to access the corresponding row of the DataFrame
peaks_df.loc[peaks_df["height"].idxmin()]

name      Pikes Peak
height    14115.0 ft
tall           False
Name: 19, dtype: object

The reduction methods are also available for a DataFrame, independently if there the data is of mixed dtypes.

In [26]:
# Define a DataFrame with a length, time and percent column, where the first two are of UnitsDtype the last one has int64 dtype
pd.DataFrame({
        "length": pd.Series(np.arange(10) * u.m, dtype="unit"),
        "time": pd.Series([0, 0, 1, 1, 1, 4, 6, 8, 8, 10] * u.s, dtype="unit"),
        "percent": [1, 3, 2, 7, 3, 0, 2, 5, 1, 9],
}).std()

length     3.0276503540974917 m
time       3.8137179293236203 s
percent                2.869379
dtype: object

## Advanced examples

### Running Diary: Advanced Calculcations, Groupings and Reductions

This example will implement a running diary. It includes randomly created measurements of `Distance` and `Time` for January 2026. Based on this the `Speed` in metric as well as imperial units will be calculated. The `Pace` will be caculated as inverse of `Speed`, normalized to minutes/kilometer and saved as pandas Timedelta. In the end the data will be grouped on the day of week and statistic will be calculated for each day. 

Define base measurements:

In [27]:
# Fix numpy random seed for consistent results
np.random.seed(0)

# Create DatetimeIndex
no_of_days: int = 31
dt_index: pd.DatetimeIndex = pd.date_range(start="2026-01-01", freq="1 D", periods=no_of_days)

# Create DataFrame with Distance and Time measurements for each day
running_df: pd.DataFrame = pd.DataFrame({
    "Distance": np.random.randint(40, 60 + 1, no_of_days)/10 * u.km,
    "Time": np.random.randint(25, 35 + 1, no_of_days) * u.min,
    },
    dtype="unit",
    index=dt_index,
)
running_df.head()

,Distance,Time
2026-01-01,5.2 km,33.0 min
2026-01-02,5.5 km,26.0 min
2026-01-03,4.0 km,28.0 min
2026-01-04,4.3 km,28.0 min
2026-01-05,4.3 km,28.0 min


Calculate derived columns:

In [28]:
# Calculate Day of Week name and Speed for each day
running_df["DoW"] = running_df.index.day_name()

# Calculate the Speed by dividing the Distance by the Time, then convert it to km/h and also to mi/h
running_df["Speed"] = (running_df["Distance"] / running_df["Time"]).units.to(u.km/u.h)
running_df["Speed_Imperial"] = running_df["Speed"].units.to("mi / h")

# Calculate the Pace as inverse of the speed and normalize it to minutes per kilometer, then convert to pandas Timedelta
running_df["Pace"] = ((running_df["Speed"] ** -1) * u.Quantity("1 km")).astype("timedelta64[ns]")

# Create a boolean column that is True if the Speed is greater than 4 km/h
running_df["Fast"] = running_df["Speed"] > 11 * u.km/u.h

running_df

,Distance,Time,DoW,Speed,Speed_Imperial,Pace,Fast
2026-01-01,5.2 km,33.0 min,Thursday,9.454545454545455 km / h,5.874782181152977 mi / h,0 days 00:06:20.769230769,False
2026-01-02,5.5 km,26.0 min,Friday,12.692307692307692 km / h,7.886634363012316 mi / h,0 days 00:04:43.636363636,True
2026-01-03,4.0 km,28.0 min,Saturday,8.571428571428571 km / h,5.32603879060572 mi / h,0 days 00:07:00,False
2026-01-04,4.3 km,28.0 min,Sunday,9.214285714285714 km / h,5.725491699901149 mi / h,0 days 00:06:30.697674418,False
2026-01-05,4.3 km,28.0 min,Monday,9.214285714285714 km / h,5.725491699901149 mi / h,0 days 00:06:30.697674418,False
2026-01-06,4.7 km,32.0 min,Tuesday,8.8125 km / h,5.475833631591507 mi / h,0 days 00:06:48.510638297,False
2026-01-07,4.9 km,25.0 min,Wednesday,11.76 km / h,7.307325220711048 mi / h,0 days 00:05:06.122448979,True
2026-01-08,5.9 km,26.0 min,Thursday,13.615384615384617 km / h,8.460207771231396 mi / h,0 days 00:04:24.406779661,True
2026-01-09,5.8 km,34.0 min,Friday,10.23529411764706 km / h,6.359916908782125 mi / h,0 days 00:05:51.724137931,False
2026-01-10,4.4 km,34.0 min,Saturday,7.764705882352942 km / h,4.8247645514898885 mi / h,0 days 00:07:43.636363636,False


Group the data by day of week and aggregate some columns:

In [ ]:
# Calculate minimum, mean and maximum speed for each day of the week
running_dow_df: pd.DataFrame = running_df.groupby("DoW").agg(
    Speed_Min=("Speed", "min"),
    Speed_Mean=("Speed", "mean"),
    Speed_Max=("Speed", "max"),
    Time_Total=("Time", "sum"),
    Fast_Count=("Fast", "sum"),
    Number_of_Runs=("Speed", "count"),
)

# Reorder the DataFrame to have the days of the week in the correct order
running_dow_df = running_dow_df.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

# Calculate the percentage of runs that were fast for each day of the week
running_dow_df["Fast_Percentage"] = (running_dow_df["Fast_Count"] / running_dow_df["Number_of_Runs"] * 100).round(2)

running_dow_df

,Speed_Min,Speed_Mean,Speed_Max,Time_Total,Fast_Count,Number_of_Runs,Fast_Percentage
DoW,,,,,,,
Monday,8.914285714285715 km / h,10.039729064039408 km / h,12.719999999999999 km / h,117.0 min,1,4,25.0
Tuesday,8.482758620689653 km / h,10.376891578249335 km / h,12.692307692307692 km / h,112.0 min,2,4,50.0
Wednesday,8.625 km / h,10.475560344827585 km / h,11.76 km / h,115.0 min,2,4,50.0
Thursday,7.0588235294117645 km / h,10.440036434154084 km / h,13.615384615384617 km / h,151.0 min,2,5,40.0
Friday,9.942857142857143 km / h,11.334091790562379 km / h,12.692307692307692 km / h,152.0 min,3,5,60.0
Saturday,7.371428571428571 km / h,9.046754540525887 km / h,10.838709677419354 km / h,160.0 min,0,5,0.0
Sunday,9.214285714285714 km / h,10.24538961038961 km / h,11.04 km / h,113.0 min,1,4,25.0


## How the Units Extension works in the background

### `UnitsDytpe`

The `UnitsDytpe` is a `pandas` `ExtensionDtype` to support unit-aware `Quantity` objects of the `astropy` package. It can be created by giving the constructor the string representation of the desired `unit` or a `unit` like `u.m` directly:

In [30]:
# Create a UnitsDtype for meters
m_dtype = pue.UnitsDtype(u.m)
m_dtype

UnitsDtype("m")

In [31]:
# The name of the dtype is how it will be represented in pandas, e.g., in the dtype attribute of a Series or DataFrame column
m_dtype.name

'unit[m]'

In [32]:
# The unit attribute gives access to the underlying astropy unit
m_dtype.unit

Unit("m")

In [33]:
# Pandas internally uses the construct_from_string method to create dtypes from strings, e.g., when creating a Series with a dtype specified as a string.
pue.UnitsDtype.construct_from_string("unit[m]")

UnitsDtype("m")

### `UnitsExtensionArray`

The `UnitsExtensionArray` is a `pandas` `ExtensionArray` that uses the `UnitsDtype` in the backend to support `astropy` `Quantity` objects in `pandas` and implement operations between `pandas` `Series`/`DataFrame` and native `Quantity` objects.

In [34]:
# Create a UnitsExtensionArray directly from a list of values and a unit
pue.UnitsExtensionArray([1, 2, 3], unit=u.m)

<UnitsExtensionArray>
[1.0 m, 2.0 m, 3.0 m]
Length: 3, dtype: unit[m]

In [35]:
# The UnitsExtensionArray of a Series can be accessed with `sr.array`, but this should be reserved for the internal API.
length_sr.array

<UnitsExtensionArray>
[1.0 m, 2.0 m, 3.0 m]
Length: 3, dtype: unit[m]